In [ ]:
import logging
import logging.config
import os

from graal.summary.blind_eval_project import BlindEvalProject
from graal.summary.llm_clients import AlbertAPIClient
from graal.utils.amendment_pre_processor import AmendmentPreProcessor

logging.config.fileConfig("logging.conf")


In [ ]:
# MAIN
from graal.summary.llm_clients import ChatGPTAPIClient


EXCEL_OUTPUT_FILE = "data/blind_eval_summaries/eval_aveugle_objets.xlsx"
PROJECT_SAVE_LOCATION = "data/blind_eval_summaries/project_eval_aveugle_objets.pkl"

METRICS = ["Concision", "Fidélité au texte", "Formulation"]

COLUMN_ORDER = ["ID", "Exposé amdt", "Corps amdt"]
for i in range(1, 3):
    COLUMN_ORDER.append(f"Objet {i}")
    for metric in METRICS:
        COLUMN_ORDER.append(f"{i} - {metric}")

DATA_FOLDER = os.getenv("DATA_FOLDER", "data")
FILE_TO_SAMPLE_FROM = (
    f"{DATA_FOLDER}/exports_lectures/PLFSS 2025/lecture_AN_avec_toutes_reponses.xlsx"
)

try:
    project = BlindEvalProject.load_from_disk(PROJECT_SAVE_LOCATION)
    logging.info(f"Project '{PROJECT_SAVE_LOCATION}' successfully loaded")
except (FileNotFoundError, EOFError):
    logging.info(f"Creating new project '{PROJECT_SAVE_LOCATION}'")
    amendments_df = AmendmentPreProcessor.load_amendments_excel(
        input_files=[FILE_TO_SAMPLE_FROM]
    )
    amendments_df = AmendmentPreProcessor.remap_columns_in_json_amendments(
        amendments_df
    )

    amendments_df = amendments_df[amendments_df["Objet amdt"].str.strip() != ""]
    amendments_df = amendments_df[
        ~amendments_df["Objet amdt"].str.contains(
            "Amendement rédactionnel|Supprimer cet article|Supprimer l'article|Amendement de coordination|Modifier l'alinea|Modifier la rédaction|irr\?",
            na=False,
        )
    ]

    project = BlindEvalProject(amendments_df=amendments_df, metrics=METRICS)

llama_70B_client = AlbertAPIClient(
    base_url=os.getenv("ETALAB_BASE_URL", "https://albert.api.etalab.gouv.fr/v1"),
    api_key=os.getenv("ETALAB_API_KEY"),
    model_name=os.getenv("ETALAB_MODEL_NAME", "meta-llama/Meta-Llama-3.1-70B-Instruct"),
)

gpt_4_turbo_client = ChatGPTAPIClient(
    model_name="gpt-4-turbo", api_key=os.getenv("OPENAI_API_KEY")
)

gpt_4o_mini_client = ChatGPTAPIClient(
    model_name="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY")
)

llm_clients = {
    # "llama-3.1-70b-instruct": llama_70B_client,
    "gpt-4-turbo": gpt_4_turbo_client,
    "gpt-4o-mini": gpt_4o_mini_client,
}

project.add_next_n_rows(5, llm_clients)

project.to_excel(
    output_file=EXCEL_OUTPUT_FILE, column_order=COLUMN_ORDER, excluded_ids=[]
)
project.dump_to_disk(output_file=PROJECT_SAVE_LOCATION)
# Open the Excel file
os.system(f'open "{EXCEL_OUTPUT_FILE}"')
project.mapping_obj_to_author